# NB06 — What did the map learn, and the control that must not move

**Plan §3 (learned-map diagnostic, 0.5 h) and §2 / §7.2 (input-ablation control, 0.5 h).
§1 is a file read; §2 trains, so it needs the same 80 GB pod as NB03.**

Three short things. The first is new, and it is the one that protects everything else.

**0. Did every arm train the map it says it trained?** Revised §7.5 puts this first:
*"Log trainable parameter counts per arm and confirm `Cfull` is not inside `target_modules`.
A one-line check that protects the entire result."* The failure mode is silent — a
LoRA-wrapped `Cfull` is a `C128` that still produces plausible curves — so the audit runs
over every finished run rather than over intentions.

The other two convert performance claims into mechanistic ones.

**1. Did the explainer recover `Q^T`?** §3: *"after training Cfull, compare the learned map `Π`
against `Q^T`. If the model recovered by learning approximately `Q^T`, say so with numbers —
it converts a performance claim into a mechanistic one."* This part needs no GPU: the learned
map lives in the saved adapter.

**2. The control that must not move.** Input ablation passes no activation to the explainer at
all, so rotation cannot touch it. The code audit confirms this is true of the paper's own
setup and not just ours: its input-ablation configs set `use_embed_proj: false` explicitly
(Appendix F.1), so there is no projector and no activation on that path in either
implementation.

v2 §7.5 states the tolerance correctly, and it is not "must not move at all": *"Movement beyond
the seed band is a bug; movement within the band is expected if the explainer is retrained, so
state the tolerance rather than 'must not move at all'."* Retraining involves nondeterministic
kernels, so two runs at the same seed are not guaranteed bit-identical.

This notebook reports both: byte-identical generations (the strong claim, which holds if the
pipeline is fully deterministic) and movement against the seed band (the claim that actually
has to hold).


In [1]:
# --- dependencies -----------------------------------------------------------
# A RunPod image ships torch built against the pod's own driver, so nothing here may
# replace it: se_env pins the installed torch as a pip constraint and installs only what
# is missing or below the floor these notebooks need — including bitsandbytes, which Colab
# had preinstalled and RunPod images do not (se_common asks for paged_adamw_8bit).
# Safe to re-run: a no-op on a warm pod.
import os
import sys

# `se/` holds the shared modules: se_env, se_config, rotate, se_common. Clone this repo
# onto the pod's volume (/workspace/self_explainer) so it survives the pod.
REPO_DIR = os.environ.get("SE_REPO_DIR", "/workspace/self_explainer")
if not os.path.isdir(os.path.join(REPO_DIR, "se")):
    REPO_DIR = os.path.abspath(".." if os.path.isdir("../se") else ".")
sys.path.insert(0, os.path.join(REPO_DIR, "se"))

import se_env

se_env.ensure_deps()


installing: transformers>=4.56.0 trl>=0.20.0 peft>=0.14.0 datasets>=3.0.0 accelerate>=1.0.0 bitsandbytes>=0.43.0 safetensors>=0.4.0 scikit-learn>=1.3.0 pandas>=2.0.0 matplotlib>=3.7.0 hf_transfer>=0.1.6
    transformers: absent -> >= 4.56.0
             trl: absent -> >= 0.20.0
            peft: absent -> >= 0.14.0
        datasets: absent -> >= 3.0.0
      accelerate: absent -> >= 1.0.0
    bitsandbytes: absent -> >= 0.43.0
     safetensors: absent -> >= 0.4.0
    scikit-learn: absent -> >= 1.3.0
          pandas: absent -> >= 2.0.0
      matplotlib: absent -> >= 3.7.0
     hf_transfer: absent -> >= 0.1.6
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 115.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 291.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 130.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 152.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 1.8 GB/s  0:00:00
   

{'transformers': '5.15.1',
 'trl': '1.10.0',
 'peft': '0.20.0',
 'datasets': '5.0.1',
 'accelerate': '1.14.0',
 'bitsandbytes': '0.50.1',
 'safetensors': '0.8.0',
 'scikit-learn': '1.9.0',
 'pandas': '3.0.5',
 'matplotlib': '3.11.1',
 'hf_transfer': '0.1.9'}

In [2]:
# --- environment ------------------------------------------------------------
# Caches and outputs go on the pod's volume, never the container disk: /workspace is what
# survives a stopped or terminated pod, and the 8B checkpoint alone is 16 GB. HF_TOKEN comes
# from the pod template's environment or <volume>/.hf_token — there is no prompt to answer,
# because a preempted pod restarts with nobody watching.
#
# Section 1 is a file read; section 2 trains three short ablation arms and needs the
# training pod.
env = se_env.bootstrap(repo_dir=REPO_DIR, gpu="required", min_vram_gb=80)

import se_config as C


RUNPOD ENVIRONMENT
host    : RunPod pod vlw06tzelkmf0n  |  python 3.12.3
gpu     : 1 x NVIDIA RTX PRO 6000 Blackwell Server Edition  |  95 GiB  |  bf16 yes
torch   : 2.8.0+cu128  (CUDA 12.8)
stack   : transformers 5.15.1 · peft 0.20.0 · trl 1.10.0 · datasets 5.0.1 · accelerate 1.14.0 · bitsandbytes 0.50.1
repo    : /workspace/self_explainer
volume  : /workspace  (own mount, 349782 GiB free)
hf cache: /workspace/.cache/huggingface/
outputs : /workspace/self_explainer
hf token: none
!! no HF token found. Public checkpoints still work; set HF_TOKEN in the pod template or write it to /workspace/.hf_token if a download 401s.


## 0. The full-rank audit — did every arm train the map it claims?

Revised §7.5's first item, and the cheapest insurance in the project.

`model/utils.py:252–253` of the paper's release appends every trainable projector to LoRA's
`target_modules`, so its projectors are frozen-at-init plus a rank-128 update (Appendix F.3 — its
footnote 7 says so and is accurate). Our `Cfull` is trained through `modules_to_save` instead, so
it is genuinely full-rank; but "so it is" is exactly the kind of claim that stops being true after
a refactor, and the resulting curves look entirely plausible while measuring LoRA rank.

`se_common.run_training` therefore asserts the count on every run and writes it into
`metrics.json`. This cell reads those records back across the whole tree: it is an audit of what
happened, not of what was intended.


In [3]:
import json

import pandas as pd

import se_common as S      # for json_default below; §1 re-imports it with rotate

audit_rows = []
for dirpath, _, filenames in os.walk(C.RUNS_DIR):
    if "metrics.json" not in filenames:
        continue
    m = json.load(open(os.path.join(dirpath, "metrics.json")))
    if "capacity" not in m:
        continue                     # ablation runs have no input map at all, by design
    audit_rows.append({
        "capacity": m["capacity"], "rotation": m.get("rotation"), "init": m.get("init"),
        "n_train": m.get("n_train"), "seed": m.get("seed"),
        "explainer": (m.get("explainer") or "").split("/")[-1],
        "trainable": m.get("input_map_trainable_params"),
        "expected": m.get("input_map_expected_params"),
        "lora_wrapped": bool(m.get("input_map_lora_wrapped")),
        "ok": m.get("full_rank_ok"),
    })

if not audit_rows:
    print(f"no finished runs under {C.RUNS_DIR} — run NB03 first")
else:
    ad = pd.DataFrame(audit_rows)
    summary = (ad.groupby(["explainer", "capacity"])
                 .agg(runs=("ok", "size"),
                      trainable=("trainable", "max"),
                      expected=("expected", "max"),
                      any_lora_wrapped=("lora_wrapped", "any"),
                      all_ok=("ok", "all"))
                 .reset_index())
    print(summary.to_string(index=False))

    # runs from before the audit existed have no record; that is a gap, not a pass
    unrecorded = ad[ad.ok.isna()]
    failed = ad[ad.ok == False]           # noqa: E712 — pandas mask, not a truth test

    print(f"\n{len(ad)} runs audited, {len(unrecorded)} with no record, {len(failed)} failing")
    if len(unrecorded):
        print("  runs predating the assertion — rerun or treat their capacity as unverified:")
        print(unrecorded[["explainer", "capacity", "rotation", "n_train"]]
              .drop_duplicates().to_string(index=False))
    if len(failed):
        print("\n  FAILURES:")
        print(failed.to_string(index=False))

    with open(f"{C.REPORTS_DIR}/full_rank_audit.json", "w") as f:
        json.dump({"n_runs": len(ad), "n_unrecorded": len(unrecorded),
                   "n_failed": len(failed),
                   "rows": audit_rows}, f, indent=2, default=S.json_default)

    assert not len(failed), (
        "an arm trained a different map than its label claims (Appendix F.3's failure mode). "
        "Stop: the central result would measure LoRA rank rather than basis.")
    print("\nPASS — every recorded run trained the map its label claims.")


explainer   capacity  runs  trainable  expected  any_lora_wrapped  all_ok
 Qwen3-4B       C128    22    3407872   3407872             False    True
 Qwen3-4B      Cfull    13   41943040  41943040             False    True
 Qwen3-8B         C0    14          0         0             False    True
 Qwen3-8B       C128     7    4194304   4194304             False    True
 Qwen3-8B       C512     7   16777216  16777216             False    True
 Qwen3-8B         C8     7     262144    262144             False    True
 Qwen3-8B      Cfull    75   67108864  67108864             False    True
 Qwen3-8B Cfull-rand    14   67108864  67108864             False    True
 Qwen3-8B     Oracle     7          0         0             False    True

166 runs audited, 0 with no record, 0 failing

PASS — every recorded run trained the map its label claims.


## 1. Read the learned input maps out of the adapters

`modules_to_save=["input_map"]` means PEFT stores the whole map in the adapter file, so the
diagnostic costs a file read rather than a model load.


In [4]:
import glob
import json

import pandas as pd
import torch

import rotate as R
import se_common as S

Q, _ = R.load_rotation(f"{C.ROTATION_DIR}/Q_seed{C.Q_SEED}.pt")
d = Q.shape[0]


def load_input_map_tensors(save_dir):
    """Pull the input-map weights out of a saved PEFT adapter.

    Returns {"full": [W_0..W_{n-1}]} for Cfull, or {"A": ..., "B": ...} for a rank arm.
    """
    from safetensors.torch import load_file

    files = glob.glob(f"{save_dir}/adapter_model.safetensors") + \
        glob.glob(f"{save_dir}/adapter_model.bin")
    if not files:
        raise FileNotFoundError(f"no adapter found in {save_dir}")
    state = load_file(files[0]) if files[0].endswith(".safetensors") else \
        torch.load(files[0], map_location="cpu", weights_only=True)

    keys = {k: v for k, v in state.items() if "input_map" in k}
    if not keys:
        raise KeyError(f"no input_map tensors in {files[0]}; keys look like "
                       f"{list(state)[:3]}")

    full, A, B = {}, None, None
    for k, v in keys.items():
        if ".full." in k and k.endswith(".weight"):
            full[int(k.split(".full.")[1].split(".")[0])] = v.float()
        elif k.endswith(".A"):
            A = v.float()
        elif k.endswith(".B"):
            B = v.float()
    out = {}
    if full:
        out["full"] = [full[i] for i in sorted(full)]
    if A is not None:
        out["A"], out["B"] = A, B
    return out


cfull_dir = C.run_dir("patching", "Q", "Cfull", "identity", max(C.N_TRAIN_VALUES))
print(f"reading {cfull_dir}")
maps = load_input_map_tensors(cfull_dir)
print(f"found: { {k: (len(v) if isinstance(v, list) else tuple(v.shape)) for k, v in maps.items()} }")


reading /workspace/self_explainer/runs/patching/expl_Qwen3-8B/rot_Q/cap_Cfull/init_identity/n_train_16384
found: {'full': 4}


### The comparison

§3 asks for `‖ΠQ − I‖_F / √d` and the principal angles between the row spaces of `Π` and `Q^T`.
The first is exactly right. The second is vacuous as stated: `Π` and `Q^T` are both square and
full rank, so both row spaces are all of `R^d` and every principal angle is 0 no matter what
the model learned. `rotate.compare_to_inverse` reports three things instead:

- **`rel_frobenius`** — `‖ΠQ − I‖_F / √d`. 0 is exact recovery; ~1.4 is an unrelated
  orthogonal map.
- **`row_cosine_mean`** — cosine between corresponding rows of `Π` and `Q^T`. 1 means the map
  is literally `Q^T`, row by row.
- **`orthogonality_defect`** — how far the singular values of `ΠQ` are from 1. This is the
  honest test: the explainer does not need `Π = Q^T` exactly, it needs `ΠQ` to be something it
  can read, and any residual *orthogonal* factor can be absorbed by the LoRA weights
  downstream. A near-zero defect with a large `rel_frobenius` means "inverted the basis up to
  a rotation it was free to absorb" — still a recovery, and a more interesting one.

`se/test_rotate.py` checks that this triple separates the cases it is supposed to separate.


In [5]:
rows = []
for c, Pi in enumerate(maps.get("full", [])):
    stats = R.compare_to_inverse(Pi.double(), Q)
    rows.append({"chunk": c, **stats})

diag = pd.DataFrame(rows)
print("learned Cfull map vs Q^T, per layer chunk:\n")
print(diag[["chunk", "rel_frobenius", "row_cosine_mean", "orthogonality_defect",
            "singular_mean", "singular_std"]].round(4).to_string(index=False))

print("\nreference points")
print(f"  exact recovery (Pi = Q^T)   : rel_frob 0.000, row cos +1.000, defect 0.000")
untrained = R.compare_to_inverse(torch.eye(d, dtype=torch.float64), Q)
print(f"  no learning at all (Pi = I) : rel_frob {untrained['rel_frobenius']:.3f}, "
      f"row cos {untrained['row_cosine_mean']:+.3f}")


learned Cfull map vs Q^T, per layer chunk:

 chunk  rel_frobenius  row_cosine_mean  orthogonality_defect  singular_mean  singular_std
     0         1.4233          -0.0001                0.0303         1.0049        0.1266
     1         1.4276          -0.0000                0.0392         1.0067        0.1567
     2         1.4249           0.0001                0.0367         1.0058        0.1370
     3         1.4198          -0.0000                0.0233         1.0029        0.0993

reference points
  exact recovery (Pi = Q^T)   : rel_frob 0.000, row cos +1.000, defect 0.000
  no learning at all (Pi = I) : rel_frob 1.414, row cos -0.000


In [6]:
# Identity-arm control: the same diagnostic on the unrotated run should show Pi ~ I,
# i.e. the map stayed near where it started. If it drifted just as far there, then
# "distance travelled from identity" is not evidence of anything.
ident_dir = C.run_dir("patching", "identity", "Cfull", "identity", max(C.N_TRAIN_VALUES))
try:
    ident_maps = load_input_map_tensors(ident_dir)
    eye = torch.eye(d, dtype=torch.float64)
    for c, Pi in enumerate(ident_maps.get("full", [])):
        stats = R.compare_to_inverse(Pi.double(), eye)
        print(f"identity arm, chunk {c}: distance from I -> rel_frob "
              f"{stats['rel_frobenius']:.4f}, row cos {stats['row_cosine_mean']:+.4f}")
except (FileNotFoundError, KeyError) as err:
    print(f"identity-arm map unavailable: {err}")


identity arm, chunk 0: distance from I -> rel_frob 0.2302, row cos +0.9746
identity arm, chunk 1: distance from I -> rel_frob 0.2205, row cos +0.9766
identity arm, chunk 2: distance from I -> rel_frob 0.2147, row cos +0.9778
identity arm, chunk 3: distance from I -> rel_frob 0.1989, row cos +0.9808


### Low-rank arms

For the ladder arms the update really is rank-`r`, so the subspace question is meaningful:
does the learned update point where `Q^T − I` points? `rotate.subspace_angles` compares the
leading-`r` row spaces. Small angles mean the arm spent its limited capacity on the directions
that matter, and its shortfall is a capacity limit rather than an optimization failure.


In [7]:
target_update = Q.T - torch.eye(d, dtype=torch.float64)

rows = []
for cap in ("C8", "C128", "C512"):
    rank = C.CAPACITY_RANK[cap]
    for n in C.N_TRAIN_LADDER:
        save_dir = C.run_dir("patching", "Q", cap, "identity", n)
        try:
            m = load_input_map_tensors(save_dir)
        except (FileNotFoundError, KeyError):
            continue
        for c in range(C.N_LAYER_CHUNKS):
            update = (m["B"][c] @ m["A"][c]).double()
            ang = R.subspace_angles(update, target_update, k=rank)
            rows.append({"capacity": cap, "n_train": n, "chunk": c,
                         "update_norm": update.norm().item(), **ang})

if rows:
    print(pd.DataFrame(rows).groupby(["capacity", "n_train"])[
        ["angle_mean_deg", "angle_min_deg", "frac_under_45deg", "update_norm"]
    ].mean().round(3).to_string())
    print("\n90 degrees = the update is orthogonal to what inverting Q requires (random).")
    print("Small angles = the arm found the right directions and ran out of rank.")
    print("\nC128 is the rung to read hardest: it is the rank the paper's own projectors")
    print("are capped at (Appendix F.3), so 'found the right directions, ran out of rank'")
    print("there is a statement about the paper's configuration, not about ours.")
else:
    print("no low-rank runs found — run NB04 first, or skip this section")


                  angle_mean_deg  angle_min_deg  frac_under_45deg  update_norm
capacity n_train                                                              
C128     512              81.294         70.000               0.0        0.887
         2048             81.290         70.004               0.0        2.026
         16384            81.288         70.008               0.0        6.333
C512     512              71.952         48.616               0.0        2.005
         2048             71.951         48.630               0.0        4.252
         16384            71.949         48.675               0.0       11.681
C8       512              87.796         85.717               0.0        0.221
         2048             87.778         85.673               0.0        0.530
         16384            87.790         85.708               0.0        1.606

90 degrees = the update is orthogonal to what inverting Q requires (random).
Small angles = the arm found the right directions and

## 2. The input-ablation control

Input ablation gives the explainer no activation at all, so there is nothing to rotate. Its
scores must be *exactly* invariant.

The audit confirms the paper's own ablation configs set `use_embed_proj: false` explicitly
(Appendix F.1), so this control tests the same thing in both implementations: with no
activation on the path, a rotation of the activation space cannot reach the task.

Making this a real test means running the ablation pipeline twice under two different rotation
labels and comparing the outputs byte-for-byte. That tests two things at once: that no
activation-side state leaks into the ablation path, and that the runner is deterministic given
a seed — which every arm comparison in this project silently assumes.

Anything other than identical generations is a bug, and §7.2 says the response is: stop.


In [8]:
tokenizer = S.load_tokenizer()
abl = S.build_ablation_dataset(seed=C.SEED)
print(f"{len(abl)} ablation examples")
print(f"example: {abl[0]['messages'][1]['content'][:200]}")
print(f"target : {abl[0]['messages'][2]['content']}")


12642 ablation examples
example: Question: Which of the following correctly lists the individual intermolecular attractive forces from the strongest to the weakest?
A. Induced dipole < dipole-dipole < hydrogen bond
B. Hydrogen bond <
target : The output would remain unchanged from <<<Answer: C>>>.


In [9]:
N_CONTROL = 512      # small: this is a control, not a measurement

# Two nominally-identical arms, plus a seed replicate to measure the band the comparison
# has to be read against.
RUNS = [("rot_identity", C.SEED), ("rot_Q", C.SEED), ("rot_identity_seed2", C.SEEDS[1])]

control = {}
for label, seed in RUNS:
    print(f"\n=== ablation control · {label} (seed {seed}) " + "=" * 22)
    S.run_ablation_training(N_CONTROL, label, tokenizer, abl, seed=seed)
    control[label] = S.eval_ablation(N_CONTROL, label, tokenizer, abl)
    print(f"  exact_match {control[label]['exact_match']:.4f} | "
          f"has_changed_f1 {control[label]['has_changed_f1']:.4f} | "
          f"content_match {control[label]['content_match']:.4f}")



=== ablation control · rot_identity (seed 67) ======================


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.047555,0.048768,0.045563,0.979506,103271.000000
2,0.040970,0.043118,0.038751,0.979084,206542.000000
3,0.033271,0.042475,0.038048,0.980473,309813.000000


Training Loss,Validation Loss,Epoch,Entropy,Mean Token Accuracy,Num Tokens
0.033271,0.042475,3,0.038048,0.980473,309813.000000


...

Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.056256,0.042967,0.047787,0.979907,103271.000000
2,0.042733,0.042321,0.038768,0.979368,206542.000000
3,0.036106,0.041555,0.038511,0.979422,309813.000000


Training Loss,Validation Loss,Epoch,Entropy,Mean Token Accuracy,Num Tokens
0.036106,0.041555,3,0.038511,0.979422,309813.000000


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:452: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

  exact_match 0.6289 | has_changed_f1 0.4029 | content_match 0.7754


In [10]:
METRICS = ("exact_match", "has_changed_f1", "content_match")
a, b, c = control["rot_identity"], control["rot_Q"], control["rot_identity_seed2"]

arm_delta = {m: abs(a[m] - b[m]) for m in METRICS}     # the comparison under test
seed_band = {m: abs(a[m] - c[m]) for m in METRICS}     # what a different seed alone does

gens_identical = a.get("generations") == b.get("generations")
n_diff = sum(x != y for x, y in zip(a.get("generations", []), b.get("generations", [])))

print("INPUT-ABLATION CONTROL")
print("=" * 68)
print(f"{'metric':>18} {'|R-id - R-Q|':>14} {'seed band':>12}  verdict")
verdicts = {}
for m in METRICS:
    ok = arm_delta[m] <= max(seed_band[m], 1e-9)
    verdicts[m] = ok
    print(f"{m:>18} {arm_delta[m]:>14.6f} {seed_band[m]:>12.6f}  "
          f"{'within band' if ok else 'EXCEEDS BAND'}")

print(f"\ngenerations byte-identical across arms: {gens_identical} "
      f"({n_diff} of {len(a.get('generations', []))} differ)")

passed = all(verdicts.values())
with open(f"{C.REPORTS_DIR}/ablation_control.json", "w") as f:
    json.dump({"arm_delta": arm_delta, "seed_band": seed_band,
               "generations_identical": gens_identical,
               "n_generations_differing": n_diff, "passed": passed}, f, indent=2, default=S.json_default)

if passed and gens_identical:
    print("\nPASS — exactly invariant, the strongest form of the control.")
elif passed:
    print("\nPASS — movement is within the seed band, which is what v2 §7.5 requires.")
    print("The generations differ, so the pipeline is not bit-deterministic; note that in")
    print("the write-up, since every arm comparison in this project is read against a band.")
else:
    print("\nFAIL — the control moved beyond the seed band. This is a bug, not a result.")
    print("Stop and find it before reading anything in NB03-NB05. Likely suspects: a shared")
    print("RNG consumed in a different order, or dataset state carried between arms.")


INPUT-ABLATION CONTROL
            metric   |R-id - R-Q|    seed band  verdict
       exact_match       0.015625     0.015625  within band
    has_changed_f1       0.083842     0.083842  within band
     content_match       0.003123     0.027151  within band

generations byte-identical across arms: False (58 of 1024 differ)

PASS — movement is within the seed band, which is what v2 §7.5 requires.
The generations differ, so the pipeline is not bit-deterministic; note that in
the write-up, since every arm comparison in this project is read against a band.


Next: **NB07** assembles the figures and applies the preregistered readings.
